In [1]:
import os, sys, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ["TORCH_NVML_DISABLED"] = "1"
torch.cuda.empty_cache()


os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm import *
import argparse


In [2]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)
ds = VQADataset(config)
df = ds.load_df()


Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/fvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/fvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/fvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.language_model.layers.16.mlp.gate_proj.weight'], processor_class=None, tokenizer_class=None, temperature=0.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', dataset_name='fvqa', pred_by='label_maxprob', split='all', suffix=''))


# Get edit_ds by eval

In [3]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)

# model
model = VQAModel(config)

# dataset
ds = VQADataset(config)
random.seed(getattr(config, "seed", 0))
ds.data = random.sample(ds.data, 50)
ds.set_dataloader(shuffle_choices=True)
ds.task_generate(model)
print(ds.task_engineer.eval(ds))

Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/fvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/fvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/fvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.language_model.layers.16.mlp.gate_proj.weight'], processor_class=None, tokenizer_class=None, temperature=0.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', dataset_name='fvqa', pred_by='label_maxprob', split='all', suffix=''))
{'uid': '4546', 'image': 'data/images/fvqa/COCO_val2014_000000014549.jpg', 'question': 'Wha

In [ ]:
pred_res_dir = os.path.join("results", "test", "pred", f"{config.model.name}", f"{config.experiment.dataset_name}")
os.makedirs(pred_res_dir, exist_ok=True)
pred_out_path = os.path.join(pred_res_dir, f"{config.experiment.task}_{config.experiment.split}.json")
edit_ds = ds.get_edits()
edit_ds.snap(out_path=pred_out_path)


# edit

In [5]:
# model = VQAModel(config)
# load the prediction set back
import json
pred_set = json.load(open(pred_out_path))
edit_ds = VQADataset(config)
edit_ds.data = pred_set
edit_ds = edit_ds.get_edits()
print(len(edit_ds.data))

2


In [6]:
import copy
model_old = copy.deepcopy(model)
# model_old_weights = copy.deepcopy(model.model.state_dict())
edit_ds.data

[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'tree',
   'label_text': 'tree',
   'label_scores': {'sandwich': {'avg_nll': 5.36581563949585,
     'sum_nll': 10.7316312789917,
     'num_tokens': 2,
     'prob': 2.184707292192121e-05},
    'car': {'avg_nll': 3.606623649597168,
     'sum_nll': 3.606623649597168,
     'num_tokens': 1,
     'prob': 0.027148432247252618},
    'tree': {'avg_nll': 0.1066236197948

In [7]:
# minimal single-batch finetune step (ft editor, no history)
editor = get_editor(config, model)
editor.generate = model.model.generate if hasattr(model, 'model') else model.generate
model.model.train()

batch = next(iter(edit_ds.loader))
tokens = model.prepare_training_batch(batch)
editor.edit(config, tokens, batch_history=None)

del tokens
torch.cuda.empty_cache()
# model_new_weights = copy.deepcopy(model.model.state_dict())
model_new = model

Finetuning module model.language_model.layers.16.mlp.gate_proj


In [8]:
edit_ds.task_generate(model_new)
edit_ds.data

[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'boat',
   'label_text': 'boat',
   'label_scores': {'tree': {'avg_nll': 20.25001335144043,
     'sum_nll': 20.25001335144043,
     'num_tokens': 1,
     'prob': 1.6052277943346565e-09},
    'car': {'avg_nll': 14.000014305114746,
     'sum_nll': 14.000014305114746,
     'num_tokens': 1,
     'prob': 8.315277909723519e-07},
    'boat': {'avg_nll': 1.41858045026

In [9]:
# model_old = copy.deepcopy(model)
# model_old.model.load_state_dict(model_old_weights)
edit_ds.task_generate(model_old)
edit_ds.data

[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'tree',
   'label_text': 'tree',
   'label_scores': {'tree': {'avg_nll': 0.09182452410459518,
     'sum_nll': 0.09182452410459518,
     'num_tokens': 1,
     'prob': 0.912401326387312},
    'car': {'avg_nll': 3.841824531555176,
     'sum_nll': 3.841824531555176,
     'num_tokens': 1,
     'prob': 0.021457622352790678},
    'boat': {'avg_nll': 2.716824531555176

## eval edits

### reliability

In [ ]:
from revlm.metrics import *

In [11]:
reliability(model_old, edit_ds)

0.0

In [12]:
reliability(model_new, edit_ds)

0.5

### generality

In [ ]:
edit_uids = [ex["uid"] for ex in edit_ds.data]
edit_uids

['4404', '580']

In [14]:
related_texts = get_t_gen_input("fvqa", edit_ds)
related_texts

{'580': ['Where can the items depicted in this image be located?',
  'In what places can the objects illustrated in this picture be found?',
  'Where are the objects shown in this image typically found?',
  'Can you tell me where the items in this image can be discovered?',
  'Where might one find the objects represented in this picture?',
  'What locations are associated with the items displayed in this image?',
  'Where do the objects featured in this image exist?',
  'In which areas can the items shown in this picture be found?',
  'Where are the objects visible in this image located?',
  'Can you specify where the items in this image can be found?'],
 '4404': ['What is typically present in this location?',
  'What might you expect to discover here?',
  'What is commonly found in this area?',
  'What could you potentially encounter in this place?',
  'What are the usual items or features in this location?',
  'What is likely to be located here?',
  'What can you anticipate finding i

In [ ]:

# repo_id = "JJoy333/RationaleVQA"
# local_root = snapshot_download(
#     repo_id=repo_id,
#     repo_type="dataset",
#     allow_patterns=["i_gen/*.parquet"],
# )
# i_gen = pd.read_parquet(os.path.join(local_root, "i_gen", f"{dataset_name}.parquet"))
# i_gen

In [16]:
# def get_i_gen_input(dataset_name: str, edit_ds, k_per_model: int = 2) -> Dict[str, List[str]]:
#     """
#     Build related_images mapping for image_generality by reading existing images only.
#     Does NOT generate new images.
#     """
#     df_full = edit_ds.load_df()
#     image2uid = dict(zip(df_full["image_path"], df_full["uid"].astype(str)))
#     edit_image_paths = {ex["image"] for ex in edit_ds.data}

#     repo_id = "JJoy333/RationaleVQA"
#     local_root = snapshot_download(
#         repo_id=repo_id,
#         repo_type="dataset",
#         allow_patterns=["i_gen/*.parquet"],
#     )
#     i_gen = pd.read_parquet(os.path.join(local_root, "i_gen", f"{dataset_name}.parquet"))
#     i_gen = i_gen[i_gen["image_path"].isin(edit_image_paths)]

#     related_images: Dict[str, List[str]] = {}
#     base_dir = Path("data/related_image") / dataset_name

#     for _, row in i_gen.iterrows():
#         image_path = row["image_path"]
#         uid = image2uid.get(image_path)
#         if uid is None:
#             continue

#         image_info_id = str(row["image_info_id"])
#         img_dir = base_dir / image_info_id
#         if not img_dir.exists():
#             continue

#         img_paths = sorted(str(p) for p in img_dir.glob("*.png"))
#         if not img_paths:
#             continue

#         # optional: apply the k_per_model-per-generator cap

#         related_images.setdefault(uid, []).extend(img_paths)

#     return related_images

In [17]:
related_images = get_i_gen_input("fvqa", edit_ds, k_per_model=2)
related_images

{'581': ['data/related_image/fvqa/val_100132/flux_0.png',
  'data/related_image/fvqa/val_100132/flux_1.png',
  'data/related_image/fvqa/val_100132/flux_2.png',
  'data/related_image/fvqa/val_100132/flux_3.png',
  'data/related_image/fvqa/val_100132/flux_4.png',
  'data/related_image/fvqa/val_100132/sd3_0.png',
  'data/related_image/fvqa/val_100132/sd3_1.png',
  'data/related_image/fvqa/val_100132/sd3_2.png',
  'data/related_image/fvqa/val_100132/sd3_3.png',
  'data/related_image/fvqa/val_100132/sd3_4.png'],
 '4404': ['data/related_image/fvqa/val_105960/flux_0.png',
  'data/related_image/fvqa/val_105960/flux_1.png',
  'data/related_image/fvqa/val_105960/sd3_0.png',
  'data/related_image/fvqa/val_105960/sd3_1.png',
  'data/related_image/fvqa/val_105960/sd3_2.png',
  'data/related_image/fvqa/val_105960/sd3_3.png',
  'data/related_image/fvqa/val_105960/sd3_4.png']}

In [ ]:
related_texts

{'580': ['Where can the items depicted in this image be located?',
  'In what places can the objects illustrated in this picture be found?',
  'Where are the objects shown in this image typically found?',
  'Can you tell me where the items in this image can be discovered?',
  'Where might one find the objects represented in this picture?',
  'What locations are associated with the items displayed in this image?',
  'Where do the objects featured in this image exist?',
  'In which areas can the items shown in this picture be found?',
  'Where are the objects visible in this image located?',
  'Can you specify where the items in this image can be found?'],
 '4404': ['What is typically present in this location?',
  'What might you expect to discover here?',
  'What is commonly found in this area?',
  'What could you potentially encounter in this place?',
  'What are the usual items or features in this location?',
  'What is likely to be located here?',
  'What can you anticipate finding i

In [19]:

# related_texts=
# related_images=
# unrelated_texts=
# unrelated_images=

# related_texts={}
# related_images={}
# for ex in edit_ds.data:
#     print(ex)
#     related_texts[ex['uid']] = [ex['question'], ex['question'], ex['question'], ex['question']]
#     related_images[ex['uid']] = [ex['image'], ex['image'], ex['image'], ex['image']]

print(image_generality(model_new, edit_ds, related_images))
print(text_generality(model_new, edit_ds, related_texts))



1.0
0.5


In [20]:
print(locality(model_old, model_new, edit_ds, sample_size=100))


0.97


In [21]:
# 20m30s
editeval(model_old = model_old,
        model_new = model_new,
        edit_ds = edit_ds,
        related_texts = related_texts,
        related_images = related_images, 
        loc_sample_size = 100)

{'reliability': 0.5,
 'text_generality': 0.5,
 'image_generality': 1.0,
 'locality': 0.97,
 'combined': 2.1366666666666667}

In [ ]:
# {'reliability': 0.5,
#  'text_generality': 0.5,
#  'image_generality': 0.5,
#  'locality': 0.84,
#  'combined': 1.8399999999999999}

In [2]:
import os

after runing revlm/run/r_gen_image.py

check the completeness of images

In [ ]:
r_gen_d = get_r_gen_input("fvqa")
remaining_sid= []
remaining_uid= []
for _, row in r_gen_d.iterrows():
    image_path = row["image_path"]
    if not os.path.exists(image_path):
        print(f"Image does not exist: {image_path}")
        remaining_sid.append(row["sid"])
        remaining_uid.append(row["uid"])



In [23]:
r_gen_d

,uid,sid,question,answer,rationale,choices,idx_choice,image_path
0,1,1_1,What is next to the man in the image?,Some bags,The image shows a man standing by some bags on...,Some bags; A dog; A bicycle; A tree,(A) Some bags\n(B) A dog\n(C) A bicycle\n(D) A...,data/r_gen/image/aokvqa/1_1.png
1,1,1_2,What is the man standing next to?,Bags on street,The image shows a man standing by some bags on...,Bags on street; Train on street; Tree on stree...,(A) Bags on street\n(B) Train on street\n(C) T...,data/r_gen/image/aokvqa/1_2.png
2,1,1_3,What is the man likely not doing?,Waiting for delivery,The image shows a man standing by some bags on...,Waiting for delivery; Standing by a train; Sit...,(A) Waiting for delivery\n(B) Standing by a tr...,data/r_gen/image/aokvqa/1_3.png
3,1,1_4,What is the skateboarder doing?,Not paying attention,The image shows a man standing by some bags on...,Not paying attention; Delivering luggage; Stan...,(A) Not paying attention\n(B) Delivering lugga...,data/r_gen/image/aokvqa/1_4.png
4,1,1_5,Where would a train not likely be found?,On the street,A train would not be on the street.,On the street; In a station; On the tracks; In...,(A) On the street\n(B) In a station\n(C) On th...,data/r_gen/image/aokvqa/1_5.png
...,...,...,...,...,...,...,...,...
132114,18195,18195_2,What color is the sign?,Yellow,The image shows a yellow sign. The lowest word...,Yellow; Red; Blue; Green,(A) Yellow\n(B) Red\n(C) Blue\n(D) Green,data/r_gen/image/aokvqa/18195_2.png
132115,18195,18195_3,What does the lowest word on the yellow sign r...,People,The image shows a yellow sign. The lowest word...,People; Animals; Speedbumps; Objects,(A) People\n(B) Animals\n(C) Speedbumps\n(D) O...,data/r_gen/image/aokvqa/18195_3.png
132116,18195,18195_4,What does the lowest word on the sign refer to?,people,The lowest word on the sign refers to people.,people; objects; animals; places,(A) people\n(B) objects\n(C) animals\n(D) places,data/r_gen/image/aokvqa/18195_4.png
132117,18195,18195_5,What does the lowest word on the sign refer to?,people,The lowest word on the sign refers to people. ...,people; animals; speedbumps; traffic,(A) people\n(B) animals\n(C) speedbumps\n(D) t...,data/r_gen/image/aokvqa/18195_5.png


In [ ]:
r_gen_d = get_r_gen_input("aokvqa")
remaining_sid= []
remaining_uid= []
remaining_rid= []
for rid, row in r_gen_d.iterrows():
    image_path = row["image_path"]
    if not os.path.exists(image_path):
        print(f"Image does not exist: {image_path}")
        remaining_sid.append(row["sid"])
        remaining_uid.append(row["uid"])
        remaining_rid.append(rid)



Image does not exist: data/r_gen/image/aokvqa/11779_1.png
Image does not exist: data/r_gen/image/aokvqa/11779_2.png
Image does not exist: data/r_gen/image/aokvqa/11779_3.png
Image does not exist: data/r_gen/image/aokvqa/11779_4.png
Image does not exist: data/r_gen/image/aokvqa/11779_5.png
Image does not exist: data/r_gen/image/aokvqa/11779_6.png
Image does not exist: data/r_gen/image/aokvqa/11780_1.png
Image does not exist: data/r_gen/image/aokvqa/11780_2.png
Image does not exist: data/r_gen/image/aokvqa/11780_3.png
Image does not exist: data/r_gen/image/aokvqa/11780_4.png
Image does not exist: data/r_gen/image/aokvqa/11780_5.png
Image does not exist: data/r_gen/image/aokvqa/11780_6.png
Image does not exist: data/r_gen/image/aokvqa/11781_1.png
Image does not exist: data/r_gen/image/aokvqa/11781_2.png
Image does not exist: data/r_gen/image/aokvqa/11781_3.png
Image does not exist: data/r_gen/image/aokvqa/11781_4.png
Image does not exist: data/r_gen/image/aokvqa/11781_5.png
Image does not

In [22]:
print(min(remaining_rid))
print(max(remaining_rid))


85448
89999


In [ ]:
edit_ds.data

unrelated_ds.df2data(pool_df)


[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'boat',
   'label_text': 'boat',
   'label_scores': {'tree': {'avg_nll': 20.25001335144043,
     'sum_nll': 20.25001335144043,
     'num_tokens': 1,
     'prob': 1.6052277943346565e-09},
    'car': {'avg_nll': 14.000014305114746,
     'sum_nll': 14.000014305114746,
     'num_tokens': 1,
     'prob': 8.315277909723519e-07},
    'boat': {'avg_nll': 1.41858045026

In [ ]:
# Test rationale_generality on a tiny dummy edit set
import copy
from revlm.metrics import rationale_generality

# Require existing model/config/edit_ds from earlier cells
try:
    model  # noqa: F401
    config  # noqa: F401
    edit_ds  # noqa: F401
except NameError:
    raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# Build a 2-sample dummy edit set from current edit_ds
_dummy = copy.deepcopy(edit_ds)
_dummy.data = _dummy.data[:2]
_dummy.set_dataloader(shuffle_choices=False)

# Map each uid to the other's uid to form a simple related_rationale
uids = [_dummy.data[i]["uid"] for i in range(len(_dummy.data))]
related_rationale = {}
if len(uids) >= 2:
    related_rationale = {uids[0]: [uids[1]], uids[1]: [uids[0]]}
else:
    # If only one example exists, just point to itself (degenerate case)
    related_rationale = {uids[0]: [uids[0]]}

print("rationale_generality:", rationale_generality(model, _dummy, related_rationale))


In [ ]:
# Test edit1_generality on the same tiny dummy edit set
import copy
from revlm.editors import get_editor
from revlm.metrics import edit1_generality, editk_bootstrap_generality

# Require existing model/config/edit_ds from earlier cells
try:
    model  # noqa: F401
    config  # noqa: F401
    edit_ds  # noqa: F401
except NameError:
    raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# Build dummy edit set (reuse 2 examples)
_dummy = copy.deepcopy(edit_ds)
_dummy.data = _dummy.data[:2]
# Keep evaluation deterministic and light
_dummy.config.n_iter = 1  # single training step inside editor.edit
_dummy.set_dataloader(shuffle_choices=True)

# Fresh base model for editing
model_old = copy.deepcopy(model)
editor = get_editor(config, model_old)
editor.generate = model_old.model.generate if hasattr(model_old, 'model') else model_old.generate

print("edit1_generality:", edit1_generality(model_old, _dummy, editor))

print("editk_bootstrap_generality:", editk_bootstrap_generality(model_old, _dummy, editor))
